In [3]:
import torch 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

In [15]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])

In [16]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("using device : ",device)

using device :  cpu


In [17]:
train_dataset = datasets.CIFAR10(
    root = "\.data",
    train = True,
    download = True,
    transform = tranform
)

In [18]:
test_dataset = datasets.CIFAR10(
    root = "\.data",
    train = False,
    download = True,
    transform = tranform
)

In [19]:
train_loader = DataLoader(
    train_dataset,
    shuffle = True,
    batch_size = 64
)

In [36]:
test_loader = DataLoader(
    train_dataset,
    shuffle = False,
    batch_size = 64
)

In [31]:
#Build CNN
class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.Conv1 = nn.Conv2d(
            3,
            32,
            3,
            1
        )

        self.Conv2 = nn.Conv2d(
            32,
            64,
            3,
            1
        )

        self.pool = nn.MaxPool2d(
            kernel_size = 2,
            stride = 2
        )

        self.fc1 = nn.Linear(
            64*6*6,
            256
        )
        self.fc2 = nn.Linear(
            256,
            128
        )

        self.fc3 = nn.Linear(
            128,
            10
        )

        self.relu = nn.ReLU()

    def forward(self,x):
        x = self.Conv1(x)
        x =self.relu(x)
        x = self.pool(x)

        x = self.Conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        x = x.view(x.size(0),-1)

        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)        
        x = self.fc3(x)

        return x


model = MyCNN().to(device)
print(model)
        

MyCNN(
  (Conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (Conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=2304, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
  (relu): ReLU()
)


In [32]:
optimizer = optim.Adam(
    model.parameters(),
    lr = 0.001
)

criterion = nn.CrossEntropyLoss()

In [34]:
# training loop

epoch = 10
for epoch in range(epoch):
    model.train()
    running_loss =0.0

    for images , labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        output = model(images)

        loss = criterion(output, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    agerage_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch+1} agerage_loss = {agerage_loss:.4f}")

Epoch 1 agerage_loss = 0.4414
Epoch 2 agerage_loss = 0.4029
Epoch 3 agerage_loss = 0.3714
Epoch 4 agerage_loss = 0.3289
Epoch 5 agerage_loss = 0.2826
Epoch 6 agerage_loss = 0.2493
Epoch 7 agerage_loss = 0.2320
Epoch 8 agerage_loss = 0.2092
Epoch 9 agerage_loss = 0.1803
Epoch 10 agerage_loss = 0.1757


In [41]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images , labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        output = model(images)

        _,predicted = torch.max(
            output,
            1
        )
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    test_acc = 100* correct/total

    print(f"accuracy = {test_acc:.2f}%")
        

accuracy = 89.41%


In [51]:
classes = (
    'plane',
    'car',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck'
)
model.eval()

with torch.no_grad():
    image, label = test_dataset[1000]
    image = image.unsqueeze(0).to(device)
    output = model(image)
    
    _, prediction = torch.max(
        output,
        dim=1
    )

print("Predicted:", classes[prediction.item()])
print("Actual:", classes[label])

Predicted: dog
Actual: dog
